In [1]:
%cd SINDyAutoencoder_hand/

/content/SINDyAutoencoder_hand


In [2]:
import os
import datetime
import pandas as pd
import numpy as np
import torch

# Assuming the architecture, loss, and data generation functions we built
# are saved in a file called `sindy_autoencoder.py`
from model import (
    SINDyAutoencoder, define_loss
)
from training import (get_lorenz_data, library_size)

def train_network_pytorch(train_data, val_data, params):
    """
    Native PyTorch implementation of the training loop with sequential thresholding.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on {device}...")

    # 1. Initialize Model and Optimizer
    model = SINDyAutoencoder(params).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])

    # 2. Prepare Data Tensors
    x_train = torch.tensor(train_data['x'], dtype=torch.float32).to(device)
    dx_train = torch.tensor(train_data['dx'], dtype=torch.float32).to(device)

    x_val = torch.tensor(val_data['x'], dtype=torch.float32).to(device)
    dx_val = torch.tensor(val_data['dx'], dtype=torch.float32).to(device)

    # Handle optional second order data
    ddx_train = torch.tensor(train_data['ddx'], dtype=torch.float32).to(device) if params['model_order'] == 2 else None
    ddx_val = torch.tensor(val_data['ddx'], dtype=torch.float32).to(device) if params['model_order'] == 2 else None

    batch_size = params['batch_size']
    num_batches = len(x_train) // batch_size

    # Initialize the mask to all ones
    mask = torch.ones((params['library_dim'], params['latent_dim']), device=device)
    params['coefficient_mask'] = mask

    print("--- STARTING MAIN TRAINING ---")
    for epoch in range(params['max_epochs']):
        model.train()
        epoch_loss = 0.0

        # Shuffle indices for mini-batching
        indices = torch.randperm(len(x_train))

        for j in range(num_batches):
            batch_idx = indices[j * batch_size : (j + 1) * batch_size]

            x_batch = x_train[batch_idx]
            dx_batch = dx_train[batch_idx]
            ddx_batch = ddx_train[batch_idx] if ddx_train is not None else None

            optimizer.zero_grad()

            # Forward pass
            network_outputs = model(x_batch, dx_batch, ddx=ddx_batch, mask=params['coefficient_mask'])

            # Calculate loss
            loss, losses, _ = define_loss(network_outputs, params)

            # Backpropagation
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        # Sequential Thresholding Step
        if params['sequential_thresholding'] and (epoch % params['threshold_frequency'] == 0) and epoch > 0:
            with torch.no_grad():
                # Get the current coefficients (Xi)
                current_coeffs = model.sindy_layer.xi.data.abs()
                # Update mask: 0 if below threshold, 1 otherwise
                new_mask = (current_coeffs >= params['coefficient_threshold']).float()
                params['coefficient_mask'] = new_mask
                active_coeffs = int(torch.sum(new_mask).item())
                print(f"THRESHOLDING: {active_coeffs} active coefficients remaining")

        # Validation & Logging
        if params['print_progress'] and (epoch % params['print_frequency'] == 0):
            model.eval()
            with torch.no_grad():
                val_outputs = model(x_val, dx_val, ddx=ddx_val, mask=params['coefficient_mask'])
                val_loss, _, _ = define_loss(val_outputs, params)
                print(f"Epoch {epoch:4d} | Train Loss: {epoch_loss/num_batches:.6f} | Val Loss: {val_loss.item():.6f}")

    print("--- STARTING REFINEMENT PHASE ---")
    # Refinement phase: freeze the mask, train without L1 regularization
    for epoch in range(params['refinement_epochs']):
        model.train()
        epoch_loss = 0.0
        indices = torch.randperm(len(x_train))

        for j in range(num_batches):
            batch_idx = indices[j * batch_size : (j + 1) * batch_size]
            x_batch, dx_batch = x_train[batch_idx], dx_train[batch_idx]
            ddx_batch = ddx_train[batch_idx] if ddx_train is not None else None

            optimizer.zero_grad()
            network_outputs = model(x_batch, dx_batch, ddx=ddx_batch, mask=params['coefficient_mask'])

            # Use the REFINEMENT loss (ignores L1 sparsity penalty)
            _, _, loss_refinement = define_loss(network_outputs, params)

            loss_refinement.backward()
            optimizer.step()
            epoch_loss += loss_refinement.item()

        if params['print_progress'] and (epoch % params['print_frequency'] == 0):
             print(f"Refinement Epoch {epoch:4d} | Train Loss: {epoch_loss/num_batches:.6f}")

    # Return final results as a dictionary to match the expected DataFrame format
    return {
        'final_loss': epoch_loss / num_batches,
        'sindy_coefficients': (model.sindy_layer.xi.data * params['coefficient_mask']).cpu().numpy()
    }

# ==============================================================================
# EXPERIMENT SETUP
# ==============================================================================
if __name__ == "__main__":

    # 1. Generate training, validation, testing data
    noise_strength = 1e-6 #[cite: 1]
    print("Generating data...")
    training_data = get_lorenz_data(1024, noise_strength=noise_strength) #[cite: 1]
    validation_data = get_lorenz_data(20, noise_strength=noise_strength) #[cite: 1]

    # 2. Set up model and training parameters
    params = {}
    params['input_dim'] = 128 #[cite: 1]
    params['latent_dim'] = 3 #[cite: 1]
    params['model_order'] = 1 #[cite: 1]
    params['poly_order'] = 3 #[cite: 1]
    params['include_sine'] = False #[cite: 1]
    params['library_dim'] = library_size(params['latent_dim'], params['poly_order'], params['include_sine'], True) #[cite: 1]

    # sequential thresholding parameters
    params['sequential_thresholding'] = True #[cite: 1]
    params['coefficient_threshold'] = 0.1 #[cite: 1]
    params['threshold_frequency'] = 500 #[cite: 1]
    params['coefficient_initialization'] = 'constant' #[cite: 1]

    # loss function weighting
    params['loss_weight_decoder'] = 1.0 #[cite: 1]
    params['loss_weight_sindy_z'] = 0.0 #[cite: 1]
    params['loss_weight_sindy_x'] = 1e-4 #[cite: 1]
    params['loss_weight_sindy_regularization'] = 1e-5 #[cite: 1]

    params['activation'] = 'sigmoid' #[cite: 1]
    params['widths'] = [64, 32] #[cite: 1]

    # training parameters
    params['epoch_size'] = training_data['x'].shape[0] #[cite: 1]
    params['batch_size'] = 8192 #[cite: 1]
    params['learning_rate'] = 1e-3 #[cite: 1]

    params['print_progress'] = True #[cite: 1]
    params['print_frequency'] = 100 #[cite: 1]

    # training time cutoffs
    params['max_epochs'] = 3001 #[cite: 1]
    params['refinement_epochs'] = 1001 #[cite: 1]

    # 3. Run training experiments
    num_experiments = 1 #[cite: 1]
    df = pd.DataFrame() #[cite: 1]

    for i in range(num_experiments):
        print(f'\nEXPERIMENT {i}') #[cite: 1]

        # Unique save name with timestamp
        params['save_name'] = 'lorenz_' + datetime.datetime.now().strftime("%Y_%m_%d_%H_%M_%S_%f") #[cite: 1]

        # Execute the PyTorch training loop
        results_dict = train_network_pytorch(training_data, validation_data, params)

        # Save results
        new_row = pd.DataFrame([{**results_dict, **params}])
        df = pd.concat([df, new_row], ignore_index=True)

    # Save to pickle format
    filename = 'experiment_results_' + datetime.datetime.now().strftime("%Y%m%d%H%M") + '.pkl' #[cite: 1]
    df.to_pickle(filename) #[cite: 1]
    print(f"\nExperiment complete. Saved to {filename}")

Generating data...

EXPERIMENT 0
Training on cuda...
--- STARTING MAIN TRAINING ---


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch    0 | Train Loss: 0.161769 | Val Loss: 0.132615
Epoch  100 | Train Loss: 0.002830 | Val Loss: 0.002658
Epoch  200 | Train Loss: 0.002738 | Val Loss: 0.002568
Epoch  300 | Train Loss: 0.002704 | Val Loss: 0.002545
Epoch  400 | Train Loss: 0.002691 | Val Loss: 0.002537
THRESHOLDING: 47 active coefficients remaining
Epoch  500 | Train Loss: 0.002699 | Val Loss: 0.002570
Epoch  600 | Train Loss: 0.002699 | Val Loss: 0.002523
Epoch  700 | Train Loss: 0.002699 | Val Loss: 0.002538
Epoch  800 | Train Loss: 0.002699 | Val Loss: 0.002523
Epoch  900 | Train Loss: 0.002698 | Val Loss: 0.002531
THRESHOLDING: 41 active coefficients remaining
Epoch 1000 | Train Loss: 0.002702 | Val Loss: 0.002551
Epoch 1100 | Train Loss: 0.002703 | Val Loss: 0.002538
Epoch 1200 | Train Loss: 0.002699 | Val Loss: 0.002535
Epoch 1300 | Train Loss: 0.002702 | Val Loss: 0.002526
Epoch 1400 | Train Loss: 0.002698 | Val Loss: 0.002528
THRESHOLDING: 37 active coefficients remaining
Epoch 1500 | Train Loss: 0.002702 

In [13]:
%ls ~/.ssh

ls: cannot access '/root/.ssh': No such file or directory


In [10]:
!git push

fatal: could not read Username for 'https://github.com': No such device or address
